In [2]:
%pip install huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
from huggingface_hub import hf_hub_download

data_path = hf_hub_download(
    repo_id="withmartian/routerbench",
    filename="routerbench_0shot.pkl",
    repo_type="dataset",
    local_dir="data/raw",
)

print(data_path)

/Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/raw/routerbench_0shot.pkl


In [4]:
import pandas as pd
df = pd.read_pickle(data_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (36497, 37)
Columns: ['sample_id', 'prompt', 'eval_name', 'WizardLM/WizardLM-13B-V1.2', 'claude-instant-v1', 'claude-v1', 'claude-v2', 'gpt-3.5-turbo-1106', 'gpt-4-1106-preview', 'meta/code-llama-instruct-34b-chat', 'meta/llama-2-70b-chat', 'mistralai/mistral-7b-chat', 'mistralai/mixtral-8x7b-chat', 'zero-one-ai/Yi-34B-Chat', 'gpt-3.5-turbo-1106|model_response', 'claude-instant-v1|model_response', 'claude-v1|model_response', 'claude-v2|model_response', 'gpt-4-1106-preview|model_response', 'meta/llama-2-70b-chat|model_response', 'mistralai/mixtral-8x7b-chat|model_response', 'zero-one-ai/Yi-34B-Chat|model_response', 'WizardLM/WizardLM-13B-V1.2|model_response', 'meta/code-llama-instruct-34b-chat|model_response', 'mistralai/mistral-7b-chat|model_response', 'gpt-3.5-turbo-1106|total_cost', 'claude-instant-v1|total_cost', 'claude-v1|total_cost', 'claude-v2|total_cost', 'gpt-4-1106-preview|total_cost', 'meta/llama-2-70b-chat|total_cost', 'mistralai/mixtral-8x7b-chat|total_cost', 'zero-

In [5]:
model = "mistralai/mistral-7b-chat"

columns = ["sample_id", "eval_name", "prompt", model, model + "|total_cost"]

display(df[columns].head(3))

print(df[model].value_counts(dropna=False))

,sample_id,eval_name,prompt,mistralai/mistral-7b-chat,mistralai/mistral-7b-chat|total_cost
0,Chinese_character_riddles.dev.0,Chinese_character_riddles,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,0.0,0.000062
1,Chinese_character_riddles.dev.1,Chinese_character_riddles,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,0.0,0.000050
2,Chinese_character_riddles.dev.10,Chinese_character_riddles,['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...,0.0,0.000051


mistralai/mistral-7b-chat
0.0     20848
1.0      7915
0.25     4696
0.75     2277
0.5       699
0.1        19
0.2        14
0.9        12
0.8         7
0.4         3
0.3         3
0.7         3
0.6         1
Name: count, dtype: int64


In [6]:
metadata_columns = ["sample_id", "prompt", "eval_name", "oracle_model_to_route_to"]

model_names = [
    column for column in df.columns
    if column not in metadata_columns and "|" not in column
]

scores = df[model_names].copy()

print("Score table shape:", scores.shape)
display(scores.head(3))

Score table shape: (36497, 11)


,WizardLM/WizardLM-13B-V1.2,claude-instant-v1,claude-v1,claude-v2,gpt-3.5-turbo-1106,gpt-4-1106-preview,meta/code-llama-instruct-34b-chat,meta/llama-2-70b-chat,mistralai/mistral-7b-chat,mistralai/mixtral-8x7b-chat,zero-one-ai/Yi-34B-Chat
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
cost_columns = [name + "|total_cost" for name in model_names]

costs = df[cost_columns].copy()

costs.columns = model_names

print("Cost table shape:", costs.shape)
display(costs.head(3))

Cost table shape: (36497, 11)


,WizardLM/WizardLM-13B-V1.2,claude-instant-v1,claude-v1,claude-v2,gpt-3.5-turbo-1106,gpt-4-1106-preview,meta/code-llama-instruct-34b-chat,meta/llama-2-70b-chat,mistralai/mistral-7b-chat,mistralai/mixtral-8x7b-chat,zero-one-ai/Yi-34B-Chat
0,0.000103,0.000099,0.004904,0.006488,0.000527,0.00973,0.000236,0.000266,0.000062,0.000166,0.000250
1,0.000073,0.000245,0.003096,0.004704,0.000205,0.01077,0.000128,0.000257,0.000050,0.000227,0.000226
2,0.000138,0.000322,0.002480,0.003728,0.000368,0.01105,0.000150,0.000232,0.000051,0.000215,0.000215


In [8]:
checks = pd.DataFrame({
    "missing_scores": scores.isna().sum(),
    "missing_costs": costs.isna().sum(),
    "scores_outside_0_1": ((scores < 0) | (scores > 1)).sum(),
    "zero_costs": (costs == 0).sum(),
    "negative_costs": (costs < 0).sum(),
})

display(checks)

,missing_scores,missing_costs,scores_outside_0_1,zero_costs,negative_costs
WizardLM/WizardLM-13B-V1.2,0,0,0,0,0
claude-instant-v1,0,0,0,0,0
claude-v1,0,0,0,0,0
claude-v2,0,0,0,0,0
gpt-3.5-turbo-1106,0,0,0,0,0
gpt-4-1106-preview,0,0,0,0,0
meta/code-llama-instruct-34b-chat,0,0,0,0,0
meta/llama-2-70b-chat,0,0,0,0,0
mistralai/mistral-7b-chat,0,0,0,0,0
mistralai/mixtral-8x7b-chat,0,0,0,0,0


In [9]:
questions = df[["sample_id", "prompt", "eval_name"]].copy()

print("Missing IDs:", questions["sample_id"].isna().sum())
print("Repeated IDs:", questions["sample_id"].duplicated().sum())
print("Missing prompts:", questions["prompt"].isna().sum())

print("First prompt type:", type(questions["prompt"].iloc[0]))
print("First prompt:", questions["prompt"].iloc[0])

Missing IDs: 0
Repeated IDs: 0
Missing prompts: 0
First prompt type: <class 'str'>
First prompt: ['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。', '古字刚分开，豆字挤进来，你先别生气，看后笑开怀。']


In [10]:
import ast

parts = ast.literal_eval(questions["prompt"].iloc[0])

print("Parsed type:", type(parts))
print("Parts:", parts)

first_question_text = "\n\n".join(parts)

print("Combined text:\n", first_question_text)

Parsed type: <class 'list'>
Parts: ['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。', '古字刚分开，豆字挤进来，你先别生气，看后笑开怀。']
Combined text:
 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。

古字刚分开，豆字挤进来，你先别生气，看后笑开怀。


In [11]:
def extract_question_text(prompt):
    try:
        parts = ast.literal_eval(prompt)
    except (ValueError, SyntaxError):
        return None

    if not isinstance(parts, list):
        return None

    if not all(isinstance(part, str) for part in parts):
        return None

    return "\n\n".join(parts).strip()


parsed_text = questions["prompt"].apply(extract_question_text)

print("Could not extract:", parsed_text.isna().sum())
print("Empty text:", (parsed_text == "").sum())

Could not extract: 0
Empty text: 0


In [12]:
questions["question_text"] = parsed_text

print("Unique question texts:", questions["question_text"].nunique())
print("Repeated question texts:", questions["question_text"].duplicated().sum())

Unique question texts: 36481
Repeated question texts: 16


In [13]:
duplicate_mask = questions["question_text"].duplicated(keep=False)

duplicate_questions = questions.loc[
    duplicate_mask, ["sample_id", "eval_name", "question_text"]
]

print("Rows in duplicate groups:", len(duplicate_questions))
print("Number of duplicate groups:", duplicate_questions["question_text"].nunique())

display(duplicate_questions.sort_values("question_text").head(6))

Rows in duplicate groups: 32
Number of duplicate groups: 16


,sample_id,eval_name,question_text
2167,chinese-remainder-theorem.dev.3,chinese-remainder-theorem,"Answer the following math question. First, pro..."
2171,chinese-remainder-theorem.dev.7,chinese-remainder-theorem,"Answer the following math question. First, pro..."
2072,bias_detection.dev.39,bias_detection,Classify the following sentence of a news arti...
2078,bias_detection.dev.44,bias_detection,Classify the following sentence of a news arti...
3140,consensus_summary.dev.58,consensus_summary,Generate a brief answer using only the provide...
3169,consensus_summary.dev.84,consensus_summary,Generate a brief answer using only the provide...


In [14]:
questions["question_group"] = pd.factorize(
    questions["question_text"], sort=False
)[0]

print("Number of question groups:", questions["question_group"].nunique())

display(
    questions.loc[
        duplicate_mask, ["sample_id", "eval_name", "question_group"]
    ].sort_values("question_group").head(6)
)

Number of question groups: 36481


,sample_id,eval_name,question_group
261,abstract2title.test.243,abstract2title,261
262,abstract2title.test.244,abstract2title,261
2072,bias_detection.dev.39,bias_detection,2071
2078,bias_detection.dev.44,bias_detection,2071
2167,chinese-remainder-theorem.dev.3,chinese-remainder-theorem,2165
2171,chinese-remainder-theorem.dev.7,chinese-remainder-theorem,2165


In [15]:
grouping_text = questions["question_text"].str.replace(
    r"\s+", " ", regex=True
).str.strip()

print("Distinct texts before:", questions["question_text"].nunique())
print("Distinct texts after:", grouping_text.nunique())

Distinct texts before: 36481
Distinct texts after: 36481


In [16]:
group_ids = questions["question_group"].drop_duplicates()

shuffled_groups = group_ids.sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("Number of groups:", len(shuffled_groups))
print(shuffled_groups.head(5))

Number of groups: 36481
0    35315
1     6888
2     3337
3    16104
4     6400
Name: question_group, dtype: int64


In [17]:
n_groups = len(shuffled_groups)

train_end = int(0.60 * n_groups)
calibration_end = train_end + int(0.10 * n_groups)
validation_end = calibration_end + int(0.10 * n_groups)

train_groups = shuffled_groups.iloc[:train_end]
calibration_groups = shuffled_groups.iloc[train_end:calibration_end]
validation_groups = shuffled_groups.iloc[calibration_end:validation_end]
test_groups = shuffled_groups.iloc[validation_end:]

print("Training groups:", len(train_groups))
print("Calibration groups:", len(calibration_groups))
print("Validation groups:", len(validation_groups))
print("Testing groups:", len(test_groups))

Training groups: 21888
Calibration groups: 3648
Validation groups: 3648
Testing groups: 7297


In [18]:
questions["split"] = "unassigned"

questions.loc[questions["question_group"].isin(train_groups), "split"] = "train"
questions.loc[questions["question_group"].isin(calibration_groups), "split"] = "calibration"
questions.loc[questions["question_group"].isin(validation_groups), "split"] = "validation"
questions.loc[questions["question_group"].isin(test_groups), "split"] = "test"

print(questions["split"].value_counts())
print("Unassigned rows:", (questions["split"] == "unassigned").sum())

split
train          21900
test            7300
validation      3649
calibration     3648
Name: count, dtype: int64
Unassigned rows: 0


In [19]:
splits_per_group = questions.groupby("question_group")["split"].nunique()

print("Groups crossing splits:", (splits_per_group > 1).sum())
print("Score indexes aligned:", questions.index.equals(scores.index))
print("Cost indexes aligned:", questions.index.equals(costs.index))

Groups crossing splits: 0
Score indexes aligned: True
Cost indexes aligned: True


In [20]:
train_mask = questions["split"] == "train"

train_questions = questions.loc[train_mask].copy()
train_scores = scores.loc[train_mask].copy()
train_costs = costs.loc[train_mask].copy()

print("Training question rows:", len(train_questions))
print("Training score shape:", train_scores.shape)
print("Training cost shape:", train_costs.shape)

Training question rows: 21900
Training score shape: (21900, 11)
Training cost shape: (21900, 11)


In [21]:
calibration_mask = questions["split"] == "calibration"

calibration_questions = questions.loc[calibration_mask].copy()
calibration_scores = scores.loc[calibration_mask].copy()
calibration_costs = costs.loc[calibration_mask].copy()

print("Calibration question rows:", len(calibration_questions))
print("Calibration score shape:", calibration_scores.shape)
print("Calibration cost shape:", calibration_costs.shape)

Calibration question rows: 3648
Calibration score shape: (3648, 11)
Calibration cost shape: (3648, 11)


In [22]:
validation_mask = questions["split"] == "validation"

validation_questions = questions.loc[validation_mask].copy()
validation_scores = scores.loc[validation_mask].copy()
validation_costs = costs.loc[validation_mask].copy()

print("Validation question rows:", len(validation_questions))
print("Validation score shape:", validation_scores.shape)
print("Validation cost shape:", validation_costs.shape)

Validation question rows: 3649
Validation score shape: (3649, 11)
Validation cost shape: (3649, 11)


In [23]:
test_mask = questions["split"] == "test"

test_questions = questions.loc[test_mask].copy()
test_scores = scores.loc[test_mask].copy()
test_costs = costs.loc[test_mask].copy()

print("Testing question rows:", len(test_questions))
print("Testing score shape:", test_scores.shape)
print("Testing cost shape:", test_costs.shape)

Testing question rows: 7300
Testing score shape: (7300, 11)
Testing cost shape: (7300, 11)


In [24]:
train_features = pd.DataFrame(index=train_questions.index)

train_features["char_length"] = train_questions["question_text"].str.len()

display(train_features.head())
print(train_features["char_length"].describe())

,char_length
0,101
3,93
4,89
5,81
6,93


count    21900.000000
mean       761.329589
std        429.631023
min         26.000000
25%        370.000000
50%        712.000000
75%       1108.000000
max       4743.000000
Name: char_length, dtype: float64


In [25]:
calibration_features = pd.DataFrame(index=calibration_questions.index)

calibration_features["char_length"] = (
    calibration_questions["question_text"].str.len()
)

validation_features = pd.DataFrame(index=validation_questions.index)

validation_features["char_length"] = (
    validation_questions["question_text"].str.len()
)

test_features = pd.DataFrame(index=test_questions.index)

test_features["char_length"] = (
    test_questions["question_text"].str.len()
)

print("Calibration features:", calibration_features.shape)
print("Validation features:", validation_features.shape)
print("Testing features:", test_features.shape)

Calibration features: (3648, 1)
Validation features: (3649, 1)
Testing features: (7300, 1)


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

toy_prompts = ["cat", "car", "猫咪"]

toy_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 2),
    lowercase=False,
)

toy_matrix = toy_vectorizer.fit_transform(toy_prompts)

toy_table = pd.DataFrame(
    toy_matrix.toarray(),
    index=toy_prompts,
    columns=toy_vectorizer.get_feature_names_out(),
)

display(toy_table)

,ar,at,ca,猫咪
cat,0.000000,0.795961,0.605349,0.0
car,0.795961,0.000000,0.605349,0.0
猫咪,0.000000,0.000000,0.000000,1.0


In [27]:
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    lowercase=False,
    min_df=3,
    max_features=20000,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_questions["question_text"]
)

print("Training TF-IDF shape:", X_train_tfidf.shape)

Training TF-IDF shape: (21900, 20000)


In [29]:
feature_names = tfidf_vectorizer.get_feature_names_out()

first_vector = X_train_tfidf[0:1].toarray().ravel()

first_weights = pd.Series(first_vector, index=feature_names)

print("First training prompt:", train_questions["question_text"].iloc[0])

display(
    first_weights[first_weights > 0]
    .sort_values(ascending=False)
    .head(10)
)

First training prompt: 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。

古字刚分开，豆字挤进来，你先别生气，看后笑开怀。


提供    0.717537
字的    0.696520
dtype: float64

In [30]:
feature_counts = pd.Series(
    X_train_tfidf.getnnz(axis=1),
    index=train_questions.index,
    name="nonzero_features",
)

print("Prompts with zero features:", (feature_counts == 0).sum())
print("Prompts with fewer than 10 features:", (feature_counts < 10).sum())

print(feature_counts.describe())

display(
    train_questions.loc[
        feature_counts < 10, ["sample_id", "eval_name"]
    ].head(10)
)

Prompts with zero features: 0
Prompts with fewer than 10 features: 206
count    21900.000000
mean      1116.937443
std        478.278998
min          1.000000
25%        694.000000
50%       1095.000000
75%       1506.000000
max       3903.000000
Name: nonzero_features, dtype: float64


,sample_id,eval_name
0,Chinese_character_riddles.dev.0,Chinese_character_riddles
3,Chinese_character_riddles.dev.11,Chinese_character_riddles
4,Chinese_character_riddles.dev.12,Chinese_character_riddles
5,Chinese_character_riddles.dev.13,Chinese_character_riddles
6,Chinese_character_riddles.dev.14,Chinese_character_riddles
7,Chinese_character_riddles.dev.15,Chinese_character_riddles
8,Chinese_character_riddles.dev.16,Chinese_character_riddles
13,Chinese_character_riddles.dev.20,Chinese_character_riddles
14,Chinese_character_riddles.dev.21,Chinese_character_riddles
17,Chinese_character_riddles.dev.24,Chinese_character_riddles


In [31]:
tfidf_vectorizer_50k = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    lowercase=False,
    min_df=3,
    max_features=50000,
)

X_train_tfidf_50k = tfidf_vectorizer_50k.fit_transform(
    train_questions["question_text"]
)

expanded_counts = X_train_tfidf_50k.getnnz(axis=1)

print("Expanded matrix shape:", X_train_tfidf_50k.shape)
print("Prompts with zero features:", (expanded_counts == 0).sum())
print("Prompts with fewer than 10 features:", (expanded_counts < 10).sum())
print("First prompt's active features:", expanded_counts[0])

Expanded matrix shape: (21900, 50000)
Prompts with zero features: 0
Prompts with fewer than 10 features: 0
First prompt's active features: 220


In [32]:
X_calibration_tfidf_50k = tfidf_vectorizer_50k.transform(
    calibration_questions["question_text"]
)

calibration_counts = X_calibration_tfidf_50k.getnnz(axis=1)

print("Calibration TF-IDF shape:", X_calibration_tfidf_50k.shape)
print("Zero-feature prompts:", (calibration_counts == 0).sum())
print("Prompts with fewer than 10 features:", (calibration_counts < 10).sum())

Calibration TF-IDF shape: (3648, 50000)
Zero-feature prompts: 0
Prompts with fewer than 10 features: 0


In [33]:
X_validation_tfidf_50k = tfidf_vectorizer_50k.transform(
    validation_questions["question_text"]
)

validation_counts = X_validation_tfidf_50k.getnnz(axis=1)

print("Validation TF-IDF shape:", X_validation_tfidf_50k.shape)
print("Zero-feature prompts:", (validation_counts == 0).sum())
print("Prompts with fewer than 10 features:", (validation_counts < 10).sum())

Validation TF-IDF shape: (3649, 50000)
Zero-feature prompts: 0
Prompts with fewer than 10 features: 0


In [34]:
X_test_tfidf_50k = tfidf_vectorizer_50k.transform(
    test_questions["question_text"]
)

test_counts = X_test_tfidf_50k.getnnz(axis=1)

print("Tesing TF-IDF shape:", X_test_tfidf_50k.shape)
print("Zero-feature prompts:", (test_counts == 0).sum())
print("Prompts with fewer than 10 features:", (test_counts < 10).sum())

Tesing TF-IDF shape: (7300, 50000)
Zero-feature prompts: 0
Prompts with fewer than 10 features: 0


In [35]:
from sklearn.preprocessing import StandardScaler

length_scaler = StandardScaler()

train_length_scaled = length_scaler.fit_transform(
    train_features[["char_length"]]
)

print("Scaled length shape:", train_length_scaled.shape)
print("Mean:", train_length_scaled.mean())
print("Standard deviation:", train_length_scaled.std())

Scaled length shape: (21900, 1)
Mean: -4.152943843712001e-17
Standard deviation: 0.9999999999999999


In [36]:
calibration_length_scaled = length_scaler.transform(
    calibration_features[["char_length"]]
)

print("Calibration length shape:", calibration_length_scaled.shape)

Calibration length shape: (3648, 1)


In [37]:
validation_length_scaled = length_scaler.transform(
    validation_features[["char_length"]]
)

print("Validation length shape:", validation_length_scaled.shape)

Validation length shape: (3649, 1)


In [38]:
test_length_scaled = length_scaler.transform(
    test_features[["char_length"]]
)

print("Testing length shape:", test_length_scaled.shape)

Testing length shape: (7300, 1)


In [39]:
from scipy.sparse import csr_matrix, hstack

X_train = hstack(
    [X_train_tfidf_50k, csr_matrix(train_length_scaled)],
    format="csr",
)

print("Combined training shape:", X_train.shape)

Combined training shape: (21900, 50001)


In [ ]:
X_calibration = hstack(
    [X_calibration_tfidf_50k, csr_matrix(calibration_length_scaled)],
    format="csr",
)

print("Combined calibration shape:", X_calibration.shape)

X_validation = hstack(
    [X_validation_tfidf_50k, csr_matrix(validation_length_scaled)],
    format="csr",
)

print("Combined validation shape:", X_validation.shape)

X_test = hstack(
    [X_test_tfidf_50k, csr_matrix(test_length_scaled)],
    format="csr",
)

print("Combined testing shape:", X_test.shape)

Combined calibration shape: (3648, 50001)
Combined training shape: (3649, 50001)
Combined training shape: (7300, 50001)


In [44]:
def check_alignment(X, question_table, score_table, cost_table):
    assert X.shape[0] == len(question_table) == len(score_table) == len(cost_table)

    assert question_table.index.equals(score_table.index)
    assert question_table.index.equals(cost_table.index)

    assert score_table.columns.tolist() == cost_table.columns.tolist() == model_names

    print("Alignment checks passed:", X.shape)

check_alignment(X_train, train_questions, train_scores, train_costs)
check_alignment(X_calibration, calibration_questions, calibration_scores, calibration_costs)
check_alignment(X_validation, validation_questions, validation_scores, validation_costs)
check_alignment(X_test, test_questions, test_scores, test_costs)

Alignment checks passed: (21900, 50001)
Alignment checks passed: (3648, 50001)
Alignment checks passed: (3649, 50001)
Alignment checks passed: (7300, 50001)


In [45]:
from pathlib import Path
from scipy.sparse import save_npz

prepared_dir = Path("data/processed/char_tfidf_50k")

prepared_dir.mkdir(parents=True, exist_ok=True)

save_npz(prepared_dir / "X_train.npz", X_train)

print("Saved:", (prepared_dir / "X_train.npz").resolve())

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/char_tfidf_50k/X_train.npz


In [46]:
save_npz(prepared_dir / "X_calibration.npz", X_calibration)
save_npz(prepared_dir / "X_validation.npz", X_validation)
save_npz(prepared_dir / "X_test.npz", X_test)

print(sorted(path.name for path in prepared_dir.glob("X_*.npz")))

['X_calibration.npz', 'X_test.npz', 'X_train.npz', 'X_validation.npz']


In [47]:
train_questions.to_pickle(prepared_dir / "train_questions.pkl")
train_scores.to_pickle(prepared_dir / "train_scores.pkl")
train_costs.to_pickle(prepared_dir / "train_costs.pkl")

print("Saved the three training tables.")

Saved the three training tables.


In [48]:
calibration_questions.to_pickle(prepared_dir / "calibration_questions.pkl")
calibration_scores.to_pickle(prepared_dir / "calibration_scores.pkl")
calibration_costs.to_pickle(prepared_dir / "calibration_costs.pkl")

print("Saved the three calibration tables.")


validation_questions.to_pickle(prepared_dir / "validation_questions.pkl")
validation_scores.to_pickle(prepared_dir / "validation_scores.pkl")
validation_costs.to_pickle(prepared_dir / "validation_costs.pkl")

print("Saved the three validation tables.")

test_questions.to_pickle(prepared_dir / "test_questions.pkl")
test_scores.to_pickle(prepared_dir / "test_scores.pkl")
test_costs.to_pickle(prepared_dir / "test_costs.pkl")

print("Saved the three testing tables.")

Saved the three calibration tables.
Saved the three validation tables.
Saved the three testing tables.


In [49]:
print(sorted(path.name for path in prepared_dir.glob("*.pkl")))

['calibration_costs.pkl', 'calibration_questions.pkl', 'calibration_scores.pkl', 'test_costs.pkl', 'test_questions.pkl', 'test_scores.pkl', 'train_costs.pkl', 'train_questions.pkl', 'train_scores.pkl', 'validation_costs.pkl', 'validation_questions.pkl', 'validation_scores.pkl']


In [50]:
import joblib

preprocessing = {
    "tfidf_vectorizer": tfidf_vectorizer_50k,
    "length_scaler": length_scaler,
    "model_names": model_names,
    "feature_order": ["tfidf", "scaled_char_length"],
}

joblib.dump(preprocessing, prepared_dir / "preprocessing.joblib")

print("Saved:", (prepared_dir / "preprocessing.joblib").resolve())

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/char_tfidf_50k/preprocessing.joblib


In [51]:
from scipy.sparse import load_npz

loaded_X_train = load_npz(prepared_dir / "X_train.npz")
loaded_train_questions = pd.read_pickle(prepared_dir / "train_questions.pkl")
loaded_train_scores = pd.read_pickle(prepared_dir / "train_scores.pkl")
loaded_train_costs = pd.read_pickle(prepared_dir / "train_costs.pkl")

loaded_preprocessing = joblib.load(prepared_dir / "preprocessing.joblib")

check_alignment(
    loaded_X_train,
    loaded_train_questions,
    loaded_train_scores,
    loaded_train_costs,
)

print("Features unchanged:", (loaded_X_train - X_train).nnz == 0)
print("Questions unchanged:", loaded_train_questions.equals(train_questions))
print("Scores unchanged:", loaded_train_scores.equals(train_scores))
print("Costs unchanged:", loaded_train_costs.equals(train_costs))

Alignment checks passed: (21900, 50001)
Features unchanged: True
Questions unchanged: True
Scores unchanged: True
Costs unchanged: True


In [52]:
prompt = loaded_train_questions["question_text"].iloc[0]

prompt_tfidf = loaded_preprocessing["tfidf_vectorizer"].transform([prompt])

prompt_length = pd.DataFrame({"char_length": [len(prompt)]})

prompt_length_scaled = loaded_preprocessing["length_scaler"].transform(
    prompt_length
)

recreated_features = hstack(
    [prompt_tfidf, csr_matrix(prompt_length_scaled)],
    format="csr",
)

difference = recreated_features - loaded_X_train[0:1]

print("Recreated shape:", recreated_features.shape)
print("Largest absolute difference:", abs(difference).max())

Recreated shape: (1, 50001)
Largest absolute difference: 5.551115123125783e-17


In [53]:
from sklearn.neighbors import NearestNeighbors

duplicate_search = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
)

duplicate_search.fit(X_train_tfidf_50k)

distances, train_positions = duplicate_search.kneighbors(
    X_validation_tfidf_50k[:3]
)

print("Closest training similarities:", 1 - distances[:, 0])
print("Matching training positions:", train_positions[:, 0])

Closest training similarities: [1.         1.         0.99768573]
Matching training positions: [ 0 36 14]


In [54]:
validation_position = 0
training_position = int(train_positions[validation_position, 0])

validation_text = validation_questions["question_text"].iloc[validation_position]
training_text = train_questions["question_text"].iloc[training_position]

print("Validation sample:", validation_questions["sample_id"].iloc[validation_position])
print("Training sample:", train_questions["sample_id"].iloc[training_position])
print("Exact text match:", validation_text == training_text)

print("\nVALIDATION TEXT:\n", validation_text)
print("\nTRAINING TEXT:\n", training_text)

Validation sample: Chinese_character_riddles.dev.10
Training sample: Chinese_character_riddles.dev.0
Exact text match: False

VALIDATION TEXT:
 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。

海边无水，种树一棵，开花没叶，会结酸果。

TRAINING TEXT:
 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。

古字刚分开，豆字挤进来，你先别生气，看后笑开怀。


In [55]:
analyzer = tfidf_vectorizer_50k.build_analyzer()

validation_grams = set(analyzer(validation_text))
training_grams = set(analyzer(training_text))

distinctive_grams = validation_grams.symmetric_difference(training_grams)

retained_distinctive_grams = distinctive_grams.intersection(
    tfidf_vectorizer_50k.vocabulary_
)

print("Distinctive character sequences:", len(distinctive_grams))
print("Retained distinctive sequences:", len(retained_distinctive_grams))
print("Retained examples:", sorted(retained_distinctive_grams)[:10])

Distinctive character sequences: 132
Retained distinctive sequences: 0
Retained examples: []


In [57]:
tfidf_vectorizer_1to4 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 4),
    lowercase=False,
    min_df=3,
    max_features=50000,
)

X_train_tfidf_1to4 = tfidf_vectorizer_1to4.fit_transform(
    train_questions["question_text"]
)

distinctive_characters = set(validation_text).symmetric_difference(
    set(training_text)
)

retained_characters = distinctive_characters.intersection(
    tfidf_vectorizer_1to4.vocabulary_
)
print("distince character length", len(distinctive_characters))
print("New training shape:", X_train_tfidf_1to4.shape)
print("Retained distinguishing characters:", sorted(retained_characters))

distince character length 25
New training shape: (21900, 50000)
Retained distinguishing characters: ['别', '古', '无', '生', '看', '结', '花', '边', '进']


In [58]:
pair_vectors = tfidf_vectorizer_1to4.transform(
    [validation_text, training_text]
)

pair_similarity = (
    pair_vectors[0:1] @ pair_vectors[1:2].T
).toarray()[0, 0]

print("Pair vector shape:", pair_vectors.shape)
print("New cosine similarity:", pair_similarity)

Pair vector shape: (2, 50000)
New cosine similarity: 0.9709257686466501


In [59]:
counts_1to4 = X_train_tfidf_1to4.getnnz(axis=1)

print("Zero-feature prompts:", (counts_1to4 == 0).sum())
print("Prompts with fewer than 10 features:", (counts_1to4 < 10).sum())
print("Minimum active features:", counts_1to4.min())
print("Mean active features:", counts_1to4.mean())

Zero-feature prompts: 0
Prompts with fewer than 10 features: 0
Minimum active features: 57
Mean active features: 1200.2761643835615
